# Suite CNT — Containers

A container holds elements by index (`Schemas.OfContainer`): a property's value, read and written through
`Visitors.OfContainer`, its elements each through a `Visitors.OfAny`. Dense, its elements are a packed array; sparse,
blocks of them; with neither, it is dense for an integer-like index and a dict for any other hashable one. These cases
cover proxies, iterators, the value objects containers hold, and snapshots.

In [ ]:
import { Builders as B, Comparison, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, root_of, same } from "./support.js";

const [Int, Text, Float] = [new S.OfScalar.Data(BigInt), new S.OfScalar.Data(String), new S.OfScalar.Data(Number)];
const store: any = new B.Store();
const Phone = store.Schemas.Object((o: any) => o.properties([
  (t: any) => t.name("number").type([Text]),
  (t: any) => t.name("label").type([Text]),
]));
const Cell = store.Schemas.Object((o: any) => o.properties([
  (t: any) => t.name("row").type([Int]),
  (t: any) => t.name("col").type([Int]),
]));
const Tags = store.Schemas.Container((c: any) => c.index([Int]).element([Text])); // dense
const Phones = store.Schemas.Container((c: any) => c.index([Text]).element([Phone])); // a dict
const Grid = store.Schemas.Container((c: any) => c.index([Cell]).element([Float])); // dense, of two indices
const Sparse = store.Schemas.Container((c: any) => c.index([Int]).element([Text]).layout("sparse"));
const Rows = store.Schemas.Container((c: any) => c.index([Int]).element([Tags])); // nested
const Card = store.Schemas.Object((o: any) => (
  o
    .name("Card")
    .ref(true)
    .pattern(String.raw`card-(\d+)`)
    .properties([
      (t: any) => t.name("tags").type([Tags]),
      (t: any) => t.name("phones").type([Phones]),
      (t: any) => t.name("grid").type([Grid]),
      (t: any) => t.name("sparse").type([Sparse]),
      (t: any) => t.name("rows").type([Rows]),
      (t: any) => t.name("name").type([Text]),
    ])
));

/** The container a property holds, as its visitor; null where it holds none (its visitor then visits nothing). */
function container(read: any): any {
  const found: any[] = [];
  read.as_value((a: any) => a.as_container((c: any) => found.push(c) && c));
  return found[0].as_schema() !== null ? found[0] : null;
}

/** An element's scalar, or null. */
function valueOfAny(e: any): unknown {
  const found: unknown[] = [];
  e.as_scalar((s: any) => found.push(s.get()) && s);
  return found.length > 0 ? found[0] : null;
}

/** A container's elements present, each with its index, as scalars. */
function scalars(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_entries((pairs: Iterable<[unknown, any]>) => {
    for (const [index, e] of pairs) out.push([index, valueOfAny(e)]);
  });
  return out;
}

/** Each position of a dense or sparse container, with its element's scalar, by its iterator. */
function walk(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_ordered((it: any) => {
    it.first();
    while (it.index() !== null) {
      const values: unknown[] = [];
      it.as_item((e: any) => e.as_scalar((s: any) => values.push(s.get()) && s));
      out.push([it.index(), values[0]]);
      it.next();
    }
  });
  return out;
}

/** Writes through a property's container. */
const into = (cb: (c: any) => unknown) => (p: any) => p.as_value((a: any) => a.as_container(cb));

## CNT-01 · Elements by index: written through a builder's container, read through a data instance's; reads never raise

In [ ]:
let card = store.Card((c: any) => (
  c
    .tags(into((t) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c")))
    .phones(into((d) => d.at("home").as_object((r: any) => r.number("1"))))
    .grid(into((g) => g.at([1n, 2n]).as_scalar(3.5)))
));
{
  const [tags, phones, grid] = [container(card.tags), container(card.phones), container(card.grid)];
  assert(tags.as_schema() === Tags && same([tags.length, phones.length, grid.length], [3, 1, 6])); // dense: every index up to the last
  assert(equal(scalars(tags), [[0n, "a"], [2n, "c"]]) && valueOfAny(tags.at(1n)) === null); // an index with nothing reads as empty
  assert(equal(walk(tags), [[0n, "a"], [1n, null], [2n, "c"]])); // an iterator visits every position of a dense container
  assert(equal(walk(grid), [[[0n, 0n], null], [[0n, 1n], null], [[0n, 2n], null], [[1n, 0n], null], [[1n, 1n], null], [[1n, 2n], 3.5]]));
  const seen: unknown[] = [];
  phones.as_dict((d: Map<unknown, unknown>) => seen.push([...d.keys()].sort()));
  tags.as_dict(() => seen.push("not a dict")); // not called: tags is dense
  phones.as_ordered(() => seen.push("not ordered")); // not called: phones is a dict
  assert(equal(seen, [["home"]]));
  phones.at("home").as_object((r: any) => seen.push(`${r.number}`) && r);
  assert(seen.at(-1) === "1");
  assert(container(store.Card((c: any) => c).tags) === null); // not set
  card.name.as_value((a: any) => a.as_container((c: any) => seen.push([c.as_schema(), c.length, c.at(0n).as_schema()])));
  assert(equal(seen.at(-1), [null, 0, null])); // a value of another kind: an empty visitor, read as an empty container
  card.name.as_value((a: any) => a.as_container((c: any) => c.at(0n).as_container((x: any) => seen.push(x.as_schema()))));
  assert(seen.at(-1) === null);
  const empty: unknown[] = [];
  card.name.as_value((a: any) => a.as_container((c: any) => c.as_entries((pairs: Iterable<unknown>) => empty.push([...pairs]))
    .as_ordered(() => empty.push("x")).as_dict(() => empty.push("x"))));
  assert(equal(empty, [[]]));
  raises(ReadOnlyError, () => tags.remove(0n), "the container is read-only; write through a builder");
  raises(ReadOnlyError, () => tags.at(0n).as_scalar("z"), "the container[0] is read-only; write through a builder");
  raises(ReadOnlyError, () => card.tags.as_value((a: any) => a.as_container(tags)), "property 'tags' is read-only; write through a builder");
  raises(ReadOnlyError, () => store.Card((c: any) => c.name((p: any) => p.as_value((a: any) => a.as_container(tags)))),
    "property 'name' does not hold a container");
  assert(repr(tags) === "<container of 2>");
  // A builder edits a copy, which it reads back; an update writes it, a clone copies it.
  const edited = store.Card(card, (c: any) => c.tags(into((t) => {
    t.remove(2n).at(1n).as_scalar("b");
    assert(t.length === 2);
  })));
  assert(edited === card && equal(scalars(container(card.tags)), [[0n, "a"], [1n, "b"]]));
  const copy = store.Card(card);
  assert(equal(scalars(container(copy.tags)), [[0n, "a"], [1n, "b"]]) && container(copy.tags) !== container(card.tags));
  // A container given as a value is written as it is; one of another schema is refused.
  const given = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(container(card.tags)))));
  assert(equal(scalars(container(given.tags)), [[0n, "a"], [1n, "b"]]));
  raises(TypeError, () => store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(container(card.grid))))),
    "property 'tags' takes a container of its schema or a Spec, got _ContainerData");
  assert(equal(scalars(container(store.Card((c: any) => c.tags(container(card.tags))).tags)), [[0n, "a"], [1n, "b"]])); // as a property's value
  const twice = store.Card((c: any) => (
    c
      .tags(into((t) => t.at(0n).as_scalar("a")))
      .tags(into((t) => t.at(1n).as_scalar("b")))
  ));
  assert(equal(scalars(container(twice.tags)), [[0n, "a"], [1n, "b"]])); // a builder reads back, and adds to, what it has written
  // Containers nest: a container's element may be a container.
  const nested = store.Card((c: any) => c.rows(into((r) => r.at(1n).as_container((t: any) => t.at(0n).as_scalar("x")))));
  const inner: unknown[] = [];
  container(nested.rows).at(1n).as_container((t: any) => inner.push(scalars(t)));
  assert(equal(inner, [[[0n, "x"]]]) && container(nested.rows).length === 2);
}

## CNT-02 · Iterators: positions before the first, at each, and past the last; a builder's extends a container and inserts into one of int indices

In [ ]:
{
  const sparse = store.Card((c: any) => c.sparse(into((c) => c.at(10n).as_scalar("j") && c.at(11n).as_scalar("k") && c.at(40n).as_scalar("z"))));
  assert(equal(walk(container(sparse.sparse)), [[10n, "j"], [11n, "k"], [40n, "z"]]) && container(sparse.sparse).length === 3);
  const moves: unknown[] = [];
  container(sparse.sparse).as_ordered((it: any) => moves.push(it.index(), it.first().index(), it.next().index(), it.last().index(),
    it.next().index(), it.next().index(), it.prev().index(), it.first().prev().index(), it.prev().index(), String(it)));
  assert(equal(moves, [null, 10n, 11n, 40n, null, null, 40n, null, null, "<iterator at None>"]));
  const calls: unknown[] = [];
  container(sparse.sparse).as_ordered((it: any) => it.last().next().as_item((e: unknown) => calls.push(e)));
  assert(calls.length === 0); // past the end of a read-only container: not called
  raises(ReadOnlyError, () => container(sparse.sparse).as_ordered((it: any) => it.first().prepend("x")),
    "the container is read-only; write through a builder");

  /** Edits a card's container through its builder's iterator. */
  const edit = (card: any, name: string, cb: (it: any) => unknown) =>
    store.Card(card, (c: any) => c[name](into((c) => c.as_ordered(cb))));

  edit(sparse, "sparse", (it) => it.last().next().as_item((e: any) => e.as_scalar("past")).first().prev().as_item(
    (e: any) => e.as_scalar("before")));
  assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "j"], [12n, "k"], [41n, "z"], [42n, "past"]]));
  edit(sparse, "sparse", (it) => it.first().append("after").prepend((e: any) => e.as_scalar("in")));
  assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "in"], [12n, "after"], [13n, "j"], [14n, "k"], [43n, "z"], [44n, "past"]]));
  const fresh = store.Card((c: any) => c.tags(into((c) => c.as_ordered(
    (it: any) => it.as_item((e: any) => e.as_scalar("first")).next().as_item((e: any) => e.as_scalar("second"))))));
  assert(equal(walk(container(fresh.tags)), [[0n, "first"], [1n, "second"]])); // an empty container extended from its ends
  edit(fresh, "tags", (it) => it.last().next().append("end").first().prev().prepend("start"));
  assert(equal(walk(container(fresh.tags)), [[0n, "start"], [1n, "first"], [2n, "second"], [3n, "end"]]));
  const blank = store.Card((c: any) => c.tags(into((c) => c.as_ordered((it: any) => it.append("only")))));
  assert(equal(walk(container(blank.tags)), [[0n, "only"]]));
  const ends = store.Card((c: any) => c.tags(into((c) => c.as_ordered(
    (it: any) => it.first().as_item((e: any) => e.as_scalar("a")).last().next().prepend("z").first().prev().append("y")))));
  assert(equal(walk(container(ends.tags)), [[0n, "y"], [1n, "a"], [2n, "z"]])); // at the ends, of an empty container and of one with elements
  const lone = store.Card((c: any) => c.tags(into((c) => c.as_ordered((it: any) => it.prepend("p")))));
  assert(equal(walk(container(lone.tags)), [[0n, "p"]]));
  const gridded = store.Card(card, (c: any) => c.grid(into((g) => g.as_ordered((it: any) => it.first().as_item((e: any) => e.as_scalar(0.5))))));
  assert(equal(walk(container(gridded.grid))[0], [[0n, 0n], 0.5]));
  raises(ValueError, () => edit(card, "grid", (it) => it.first().append(1.0)),
    "property 'grid': only a dense or sparse container of int indices has positions to insert at; write through at(index)");
}

## CNT-03 · Value objects in containers belong to the object holding the container: edited in place, copied by a clone, removed with their element

In [ ]:
{
  const owner = store.Card((c: any) => c.phones(into((d) => d.at("home").as_object((r: any) => r.number("1").label("h"))
    && d.at("work").as_object((r: any) => r.number("2")))));
  const held = new Map<unknown, any>();
  container(owner.phones).as_dict((d: Map<unknown, any>) => d.forEach((e, k) => held.set(k, e)));
  const home: any[] = [];
  held.get("home").as_object((r: any) => home.push(r) && r);
  assert(Proxies.owner_of(home[0]) === owner && `${home[0].label}` === "h");
  store.Card(owner, (c: any) => c.phones(into((d) => d.at("home").as_object((r: any) => r.label("x")))));
  const again: any[] = [];
  container(owner.phones).at("home").as_object((r: any) => again.push(r) && r);
  assert(again[0] === home[0] && `${home[0].label}` === "x" && `${home[0].number}` === "1"); // edited in place
  const twin = store.Card(owner);
  const copied: any[] = [];
  container(twin.phones).at("home").as_object((r: any) => copied.push(r) && r);
  assert(copied[0] !== home[0] && Proxies.owner_of(copied[0]) === twin && `${copied[0].number}` === "1");
  store.Card(owner, (c: any) => c.phones(into((d) => d.remove("home"))));
  assert(container(owner.phones).length === 1 && Proxies.owner_of(home[0]) === owner);
  const Kit = store.Schemas.Object((o: any) => o.properties([(t: any) => t.name("parts").type([Tags])])); // a value object holding a container
  const Box = store.Schemas.Object((o: any) => o.name("Box").ref(true).properties([(t: any) => t.name("kit").type([Kit])]));
  const box = store.Box((b: any) => b.kit((p: any) => p.as_object((k: any) => k.parts(into((t) => t.at(0n).as_scalar("bolt"))))));
  const twinBox = store.Box(box);
  const parts: unknown[] = [];
  twinBox.kit.as_object((k: any) => {
    k.parts.as_value((a: any) => a.as_container((t: any) => parts.push(scalars(t))));
    return k;
  });
  assert(equal(parts, [[[0n, "bolt"]]])); // copied with the value object that holds it
  const moved = store.Card((c: any) => c.tags(into((t) => t.at(0n).as_scalar("a"))));
  store.Card(moved, (c: any) => c.tags(into((t) => t.as_ordered((it: any) => it.first().prepend("z")))));
  assert(equal(walk(container(moved.tags)), [[0n, "z"], [1n, "a"]]));
}

## CNT-04 · Snapshots: a dense container as an array, nested or flat for several indices, a sparse one as blocks, a dict by text as a mapping, entries otherwise; read back, and over what a store holds

In [ ]:
card = store.Card((c: any) => (
  c
    .tags(into((t) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c")))
    .phones(into((d) => d.at("home").as_object((r: any) => r.number("1"))))
    .grid(into((g) => g.at([1n, 2n]).as_scalar(3.5) && g.at([0n, 0n]).as_scalar(0.25)))
    .sparse(into((s) => s.at(3n).as_scalar("x") && s.at(4n).as_scalar("y") && s.at(9n).as_scalar("z")))
    .rows(into((r) => r.at(0n).as_container((t: any) => t.at(1n).as_scalar("n"))))
));
const written = root_of(Plain.ToPlain(store).OfObject(Card, card));
assert(container(card.grid).length === 6); // the box its indices span
assert(equal(written, map({ tags: ["a", null, "c"], phones: { home: { number: "1" } }, grid: [[0.25, null, null], [null, null, 3.5]],
  sparse: [{ at: 3n, items: ["x", "y"] }, { at: 9n, items: ["z"] }], rows: [[null, "n"]] })));
const flat = root_of(Plain.ToPlain(store, { flat: true }).OfObject(Card, card));
assert(equal(flat.get("grid"), map({ shape: [2n, 3n], items: [0.25, null, null, null, null, 3.5] })));
assert(JSON.ToJSON(store, { flat: true }).OfObject(Card, card).includes('"grid": {"shape": [2, 3]')
  && YAML.ToYAML(store, { flat: true }).OfObject(Card, card).includes("shape:"));
const other: any = new B.Store(); // another store of the same schemas
other.register(Card);
for (const snapshot of [Plain.ToPlain(store).OfObject(Card, card), Plain.ToPlain(store, { flat: true }).OfObject(Card, card)]) {
  const back = Plain.FromPlain(other).OfObject(Card, snapshot) as any; // the loader takes either form
  assert(equal(root_of(Plain.ToPlain(other).OfObject(Card, back)), written));
}
assert(Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) === card); // reloading: an update
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, card)), written));
// Any other index is written as entry-like elements: a dict by bytes, a dense container by text, a sparse one by text.
const ByBytes = store.Schemas.Container((c: any) => c.index([new S.OfScalar.Data(Uint8Array)]).element([Int]));
const Sorted = store.Schemas.Container((c: any) => c.index([Text]).element([Int]).layout("dense"));
const Keyed = store.Schemas.Container((c: any) => c.index([Cell]).element([Int]).layout("sparse"));
const Mixed = store.Schemas.Object((o: any) => o.properties([
  (t: any) => t.name("k").type([Text]),
  (t: any) => t.name("n").type([Int]),
]));
const ByMixed = store.Schemas.Container((c: any) => c.index([Mixed]).element([Int]));
const Note = store.Schemas.Object((o: any) => (
  o
    .name("Note")
    .ref(true)
    .properties([
      (t: any) => t.name("b").type([ByBytes]),
      (t: any) => t.name("s").type([Sorted]),
      (t: any) => t.name("k").type([Keyed]),
      (t: any) => t.name("m").type([ByMixed]),
    ])
));
const note = store.Note((n: any) => (
  n
    .b(into((c) => c.at(new Uint8Array([1])).as_scalar(1n)))
    .s(into((c) => c.at("b").as_scalar(2n) && c.at("a").as_scalar(1n)))
    .k(into((c) => c.at([0n, 1n]).as_scalar(1n) && c.at([0n, 2n]).as_scalar(2n) && c.at([3n, 0n]).as_scalar(3n)))
    .m(into((c) => c.at(["x", 1n]).as_scalar(5n)))
));
const noted = root_of(Plain.ToPlain(store).OfObject(Note, note));
assert(equal(noted, map({ b: [{ index: "AQ==", element: 1n }], s: [{ index: "a", element: 1n }, { index: "b", element: 2n }],
  k: [{ at: { row: 0n, col: 1n }, items: [1n, 2n] }, { at: { row: 3n, col: 0n }, items: [3n] }],
  m: [{ index: { k: "x", n: 1n }, element: 5n }] })));
other.register(Note);
assert(equal(root_of(Plain.ToPlain(other).OfObject(Note, Plain.FromPlain(other).OfObject(Note, Plain.ToPlain(store).OfObject(Note, note)) as any)), noted));
// A union's branch, or an intersection's part, may hold a container.
const Either = store.Schemas.Union((u: any) => u.branches([
  (t: any) => t.name("tags").type([Tags]),
  (t: any) => t.name("word").type([Text]),
]));
const Choice = store.Schemas.Object((o: any) => o.name("Choice").ref(true).properties([(t: any) => t.name("pick").type([Either])]));
other.register(Choice);
const choice = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(1n).as_scalar("q"))))));
const picked = root_of(Plain.ToPlain(store).OfObject(Choice, choice));
assert(equal(picked, map({ pick: { tags: [null, "q"] } })));
assert(equal(root_of(Plain.ToPlain(other).OfObject(Choice, Plain.FromPlain(other).OfObject(Choice, Plain.ToPlain(store).OfObject(Choice, choice)) as any)), picked));
store.Choice(choice, (c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(2n).as_scalar("r"))))));
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, choice)), map({ pick: { tags: [null, "q", "r"] } }))); // its container edited
const sameBranch = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union(
  (u: any) => u.tags((t: any) => t.at(0n).as_scalar("p")).tags((t: any) => t.at(1n).as_scalar("q")))))); // read back, then written again
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, sameBranch)), map({ pick: { tags: ["p", "q"] } })));
const givenBranch = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(container(card.tags))))));
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, givenBranch)), map({ pick: { tags: ["a", null, "c"] } }))); // a container given
raises(TypeError, () => store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(1n))))),
  "branch 'tags' takes a container of its schema or a Spec, got int");
raises(ReadOnlyError, () => container(card.tags)((t: unknown) => t), "the container is read-only; write through a builder");
// An entry's property may hold a container: entries equal by their containers are one, in a set.
const Sets = store.Schemas.Relation((r: any) => (
  r
    .name("Sets")
    .links([
      (t: any) => t.name("a"),
      (t: any) => t.name("b"),
    ])
    .properties([
      (t: any) => t.name("tags").type([Tags]),
      (t: any) => t.name("grid").type([Grid]),
    ])
    .uniques([[]])
));
const Marks = store.Schemas.Object((o: any) => (
  o
    .name("Marks")
    .ref(true)
    .pattern(String.raw`marks-(\d+)`)
    .adjacencies([
      (t: any) => t.name("sets").relation([Sets]).me("a"),
      (t: any) => t.name("set_by").relation([Sets]).me("b"),
    ])
));
other.register(Marks);
other.register(Sets);
const two = (t: any) => t.at(0n).as_scalar("a") && t.at(1n).as_scalar("b");
const cell = (g: any) => g.at([0n, 1n]).as_scalar(1.5);
const target = store.Marks((m: any) => m);
const marks = store.Marks((m: any) => m.sets((adj: any) => (
  adj
    .insert((e: any) => e.b(target).tags(into(two)).grid(into(cell)))
    .insert((e: any) => e.b(target).tags(into(two)).grid(into(cell)))
)));
const row = { b: { $ref: "Marks/marks-1", $schema: "Marks" }, tags: ["a", "b"], grid: [[null, 1.5]] };
assert(marks.sets.length === 1 && equal(root_of(Plain.ToPlain(store).OfObject(Marks, marks)), map({ sets: [row] })));
const doubled = map({ root: "Marks/marks-2", objects: { "Marks/marks-2": { sets: [row, row] }, "Marks/marks-1": {} } });
assert((Plain.FromPlain(other).OfObject(Marks, doubled) as any).sets.length === 1); // read once, by value
assert(container(note.s).length === 2); // a dense container by text: the elements present
const emptied = Plain.FromPlain(other).OfObject(Note, map({ root: "Note/e", objects: { "Note/e": { b: [{ index: "AQ==" }], s: [] } } })) as any;
assert(equal(root_of(Plain.ToPlain(other).OfObject(Note, emptied)), map({ b: [], s: [] }))); // an element absent; empty containers
const branch: any[] = [];
choice.pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
store.Choice((c: any) => c).pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
assert(branch[0].length === 3 && branch[1].as_schema() === null); // a union value not set: its branch reads as empty

## CNT-05 · Malformed containers in snapshots are refused, with a path

In [ ]:
{
  const refused = (name: string, value: unknown, message: string) =>
    raises(DecodeError, () => Plain.FromPlain(store).OfObject(Card, map({ root: "Card/card-900", objects: { "Card/card-900": { [name]: value } } })), message);
  const at = '$.objects["Card/card-900"]';
  refused("tags", { a: 1n }, `${at}.tags: a dense container must be a list, got dict`);
  refused("tags", [1n], `${at}.tags[0]: expected str, got int`);
  refused("grid", [[1.0], 2.0], `${at}.grid[1]: a container of 2 indices nests 2 lists`);
  refused("grid", { shape: [2n], items: [1.0] }, `${at}.grid: a flat container's shape is 2 sizes, and its items as many as they make`);
  refused("grid", { shape: [1n, 2n], items: [1.0] }, `${at}.grid: a flat container's shape is 2 sizes`);
  refused("grid", { shape: "2, 3", items: [] }, `${at}.grid: a flat container's shape is 2 sizes`);
  refused("grid", { shape: [1n, 1n], items: [true] }, `${at}.grid.items[0]: expected float, got bool`);
  refused("sparse", { at: 1n }, `${at}.sparse: a sparse container must be a list of blocks, got dict`);
  refused("sparse", [{ at: 1n }], `${at}.sparse[0]: a block is {'at': index, 'items': [element, ...]}`);
  refused("sparse", [{ at: "1", items: [] }], `${at}.sparse[0].at: expected int, got str`);
  refused("phones", [], `${at}.phones: a container keyed by text must be a mapping, got list`);
  refused("phones", { home: 1n }, `${at}.phones.home: a value object must be a mapping, got int`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { m: {} } } })),
    '$.objects["Note/n"].m: a container must be a list, got dict');
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { m: [{ element: 1n }] } } })),
    `$.objects["Note/n"].m[0]: an entry of a container is {'index': index, 'element': element}`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, map({ root: "Note/n", objects: { "Note/n": { k: [{ at: { row: 1n }, items: [1n] }] } } })),
    `$.objects["Note/n"].k[0].at: an index has every property of its value object; ['col'] are not set`);
}
const Deep = store.Schemas.Object((o: any) => o.properties([(t: any) => t.name("cell").type([Cell])]));
const Odd = store.Schemas.Container((c: any) => c.index([Deep]).element([Int])); // entries: its index is neither ordered nor hashable
const Rare = store.Schemas.Object((o: any) => o.name("Rare").ref(true).properties([(t: any) => t.name("odd").type([Odd])]));
raises(DecodeError, () => Plain.FromPlain(store).OfObject(Rare, map({ root: "Rare/r", objects: { "Rare/r": { odd: [{ index: {}, element: 1n }] } } })),
  `$.objects["Rare/r"].odd[0].index: a container whose index is neither ordered nor hashable is not read from snapshots yet`);

## CNT-06 · An index is checked against the index type: a scalar of its type, or a tuple of a value object index's properties; a dense container's is not negative

In [ ]:
{
  const at = (name: string, index: unknown) => store.Card((c: any) => c[name](into((c) => c.at(index))));
  raises(TypeError, () => at("tags", "0"), "property 'tags': an index is a int, got str");
  raises(TypeError, () => at("tags", true), "property 'tags': an index is a int, got bool");
  raises(TypeError, () => at("grid", [1n]), "property 'grid': an index is a tuple of int, int, got [1]");
  raises(TypeError, () => at("grid", [1n, "2"]), "property 'grid': an index is a tuple of int, int, got [1, '2']");
  raises(ValueError, () => at("tags", -1n), "property 'tags': a dense container's index is not negative, got -1");
  raises(ValueError, () => at("grid", [0n, -1n]), "property 'grid': a dense container's index is not negative, got [0, -1]");
  store.builder("Card").sparse(into((c) => c.at(-5n).as_scalar("ok"))); // sparse: any int
  const deep: any[] = [];
  store.builder("Rare").odd(into((c) => deep.push(c)));
  raises(TypeError, () => deep[0].at(1n), "property 'odd': an index is a value of its index type, got int");
  // An index of a value object that is neither ordered nor hashable: given as a value object, written as one.
  const Keyed = store.Schemas.Object((o: any) => (
    o
      .name("Keyed")
      .ref(true)
      .properties([
        (t: any) => t.name("key").type([Deep]),
        (t: any) => t.name("odd").type([Odd]),
      ])
  ));
  let keyed = store.Keyed((k: any) => k.key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(2n))))));
  const key: any[] = [];
  keyed.key.as_object((k: any) => key.push(k) && k);
  keyed = store.Keyed(keyed, (k: any) => k.odd(into((c) => c.at(key[0]).as_scalar(7n))));
  assert(equal(root_of(Plain.ToPlain(store).OfObject(Keyed, keyed)).get("odd"), map([{ index: { cell: { row: 1n, col: 2n } }, element: 7n }])));
}

## CNT-07 · Validation and comparison: each element checked at its index; containers compared by index and element, by value in uniqueness; different index or element types incomparable

In [ ]:
{
  const put = (pairs: [unknown, unknown][], write: (e: any, v: any) => unknown) =>
    into((c) => pairs.map(([i, v]) => write(c.at(i), v)));
  const Line = store.Schemas.Object((o: any) => o.properties([(t: any) => t.name("number").type([Text])]));
  const Words = store.Schemas.Container((c: any) => c.index([Int]).element([Text]));
  const Book = store.Schemas.Container((c: any) => c.index([Text]).element([Line]));
  const Table = store.Schemas.Container((c: any) => c.index([Cell]).element([Float]));
  const Nest = store.Schemas.Container((c: any) => c.index([Int]).element([Words]));
  const ByDeep = store.Schemas.Container((c: any) => c.index([Deep]).element([Int]));
  const Labels = store.Schemas.Relation((r: any) => (
    r
      .name("Labels")
      .links([
        (t: any) => t.name("shelf"),
        (t: any) => t.name("label"),
      ])
      .properties([
        (t: any) => t.name("words").type([Words]),
        (t: any) => t.name("deep").type([ByDeep]),
        (t: any) => t.name("table").type([Table]),
        (t: any) => t.name("line").type([Line]),
      ])
      .uniques("label")
  ));
  const Shelf = store.Schemas.Object((o: any) => (
    o
      .name("Shelf")
      .ref(true)
      .pattern(String.raw`shelf-(\d+)`)
      .properties([
        (t: any) => t.name("words").type([Words]),
        (t: any) => t.name("book").type([Book]),
        (t: any) => t.name("table").type([Table]),
        (t: any) => t.name("nest").type([Nest]),
        (t: any) => t.name("deep").type([ByDeep]),
        (t: any) => t.name("key").type([Deep]),
      ])
      .adjacencies([
        (t: any) => t.name("labels").relation([Labels]).me("shelf"),
        (t: any) => t.name("labelled").relation([Labels]).me("label"),
      ])
  ));
  const held = (read: any): any => {
    const found: any[] = [];
    read.as_object((k: any) => found.push(k) && k);
    return found[0];
  };
  const shelf = (words: [unknown, unknown][], book: [unknown, unknown][], row: bigint): any => {
    const made = store.Shelf((s: any) => (
      s
        .words(put(words, (e, v) => e.as_scalar(v)))
        .book(put(book, (e, v) => e.as_object((r: any) => r.number(v))))
        .table(put([[[0n, 1n], 0.5]], (e, v) => e.as_scalar(v)))
        .nest(put([[0n, "x"]], (e, v) => e.as_container((w: any) => w.at(1n).as_scalar(v))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(row).col(0n)))))
    ));
    return store.Shelf(made, (s: any) => s.deep(put([[held(made.key), 1n]], (e, v) => e.as_scalar(v))));
  };
  const compare = (x: any, y: any) =>
    new Comparison.OfObject(Shelf, x, { value: true }).compare(new Comparison.OfObject(Shelf, y, { value: true }));
  const book: [unknown, unknown][] = [["home", "1"], ["work", "2"]];
  const a = shelf([[0n, "a"], [2n, "c"]], book, 1n);
  assert(compare(a, shelf([[0n, "a"], [2n, "c"]], [["work", "2"], ["home", "1"]], 1n)) === 0); // a dict's order is not compared
  assert(compare(a, shelf([[0n, "a"], [1n, "c"]], book, 1n)) === null); // an element at another index
  assert(compare(a, shelf([[0n, "a"], [2n, "x"]], book, 1n)) === null); // another element
  assert(compare(a, shelf([[0n, "a"]], book, 1n)) === null); // fewer elements
  assert(compare(a, shelf([[0n, "a"], [2n, "c"]], book, 2n)) === null); // another value object index
  const words = container(a.words);
  const Other = store.Schemas.Container((c: any) => c.index([Text]).element([Text])); // another index type
  const Counts = store.Schemas.Container((c: any) => c.index([Int]).element([Int])); // another element type
  assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Words, container(a.words))) === 0);
  assert(new Comparison.OfContainer(Words).compare(new Comparison.OfContainer(Words)) === 0); // absent, both
  assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Words)) === null);
  assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Other)) === null);
  assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Counts)) === null);
  // Uniqueness compares containers by value: two entries of one shelf with equal containers name one label.
  const [b, c] = [shelf([], [], 1n), shelf([], [], 1n)];
  const labelled = (target: any, word: string) => (x: any) => x.label(target)
    .words(put([[0n, word]], (e, v) => e.as_scalar(v)))
    .deep(put([[held(a.key), 2n]], (e, v) => e.as_scalar(v))).table(put([[[0n, 1n], 0.5]], (e, v) => e.as_scalar(v)));
  store.Shelf(a, (s: any) => s.labels((r: any) => r.insert(labelled(b, "w")).insert(labelled(c, "w")).insert(labelled(c, "v"))));
  const unique = "unique(label) violated: entries agreeing on ['deep', 'line', 'shelf', 'table', 'words'] differ on ['label']"; // no line set
  assert(equal(Validators.Validate(store).OfObject(Shelf, a), [unique]));
  // Validation checks each element, at its index.
  store.Schemas.Container(Words, (c: any) => c.element([Int]));
  store.Schemas.Container(Table, (c: any) => c.element([Int]));
  store.Schemas.Object(Line, (o: any) => o.properties([(t: any) => t.name("number").type([Int])]));
  assert(equal(Validators.Validate(store).OfObject(Shelf, a), [
    "Shelf#0.words[0]: expected int, got str", "Shelf#0.words[2]: expected int, got str",
    "Shelf#0.book['home'].number: expected int, got str", "Shelf#0.book['work'].number: expected int, got str",
    "Shelf#0.table[[0, 1]]: expected int, got float", "Shelf#0.nest[0][1]: expected int, got str",
    "Shelf#0.labels[0].words[0]: expected int, got str", "Shelf#0.labels[0].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[1].words[0]: expected int, got str", "Shelf#0.labels[1].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[2].words[0]: expected int, got str", "Shelf#0.labels[2].table[[0, 1]]: expected int, got float",
    unique]));
}